In [ ]:
# install dependencies
%pip install anthropic python-dotenv openai openrouter

In [ ]:
# load env variables
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("ANTHROPIC_API_KEY")[:10])
print(os.getenv("DEEPSEEK_API_KEY")[:10])

In [12]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

model = "deepseek-chat"
base_url = "https://api.deepseek.com"

messages = []

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=base_url
)

def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": text
    })
    return messages

def add_assistant_message(messages, text):
    messages.append({
        "role": "assistant",
        "content": text
    })

def chat(messages, system_prompt=None):
    
    if system_prompt:
        request_messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            *messages
        ]
    else:
        request_messages = messages
        
    response = client.chat.completions.create(
        model=model,
        messages=request_messages
    )

    answer = response.choices[0].message.content
    add_assistant_message(messages, answer)
    return answer


# while True:
#     user_input = input("You: ")

#     if user_input.lower() in ["quit", "exit", "bye"]:
#         break

#     add_user_message(messages, user_input)

#     reply = chat(messages)
#     print("Assistant:", reply)

In [ ]:
# implementing the system prompt
message = add_user_message([], "what is 2 + 2")
reply = chat(message, "yor are sarcastic friend. Answer accordingly")
reply

In [14]:
# response streaming
def chat_stream(messages, system_prompt=None):
    request_messages = (
        [
            {
                "role": "system",
                "content": system_prompt
            },
            *messages
        ]
        if system_prompt
        else messages
    )
    stream = client.chat.completions.create(
        model = model,
        messages = request_messages,
        stream = True
    )
    answer = ""
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)
            answer += content
    print()
    add_assistant_message(messages, answer)
    return answer

In [ ]:
messages = []
add_user_message(messages, "tell me a short joke in Marathi!")
reply = chat_stream(messages)
reply

In [17]:
# structured data
import json
def chat_json(messages, system_prompt=None):
    request_messages = (
        [
            {
                "role": "system",
                "content": system_prompt
            },
            *messages
        ]
        if system_prompt
        else messages
    )
    response = client.chat.completions.create(
        model = model,
        messages = request_messages,
        stop = ["\n\n"] # optional
    )
    content = response.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {
            "raw_response": content
        }

In [ ]:
messages = []
add_user_message(
    messages,
    """ Return the answer as JSON:
    {
        "question":"",
        "answer":""
    }
    W
    """
)